# Hivemind on LongMemEval (search only): Qwen3-Embedding-8B, all questions

This notebook measures Hivemind's search on all 470 answerable LongMemEval_S questions (the 30 abstention questions are left out) with Qwen3-Embedding-8B at dim 2000, the model production is assumed to run. That is about 18,000 texts and 35 million tokens, roughly 9 to 10 hours of embedding on two T4s. Kaggle stops a session after 12 hours, so the notebook stops itself after `TIME_BUDGET_HOURS` (10 by default) and saves the vectors it has. A second run picks up from there.

**Before you run it:** in the right-hand panel, set *Accelerator* to **GPU T4 x2** and switch **Internet** on.

**First run:** click **Save Version**, choose **Save & Run All (Commit)**, then **Save**. It runs in the background and you can close the tab. Set `REF` in the first code cell to measure a branch or tag other than `main`.

**When it finishes,** open the version and go to **Output**.
- If there is a `longmemeval-qwen3-8b.md`, it is done. Download it, `longmemeval-qwen3-8b.json` and any `longmemeval-qwen3-8b-notes.txt` (texts embedded in fp32 or shortened).
- If there is a `longmemeval-qwen3-8b-progress.txt` instead, it ran out of time; the file says how far it got. Open the notebook in the editor, click **Add Input**, choose **Your Work**, then **Notebooks**, and add this notebook's latest version (its output holds the vectors, in `lme-cache/`). If Kaggle doesn't offer this notebook itself, use **File → Make a copy** and add the first notebook's output to the copy. Then **Save & Run All (Commit)** again. Vectors from every attached output are merged in and not redone.

Each run uses up to about 11.5 of Kaggle's 30 free GPU hours a week, so two runs fit in one week.

**Running it again after the harness changes** (for example a new variant, such as `no_instruction`, which only needs the 470 questions embedded again): reuse the vectors from the run that finished.
1. Open this notebook in the editor. In the right-hand panel under **Input**, check what is attached. Add the version whose output has `longmemeval-qwen3-8b.md` if it isn't there yet: **Add Input**, **Your Work**, **Notebooks**, this notebook, then pick that version. Older versions can stay attached; their vectors are merged too.
2. In the last code cell, set `MAX_NEW_TEXTS = 1000`. If the cache isn't found, the run then stops within a few minutes with a message instead of embedding everything again for 10 hours.
3. **Save & Run All (Commit)**. Near the top of the output it prints a line like `470 questions, 470 texts to embed, 18064 already cached`. The run takes about 15 minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q uv
REF = "main"  # the branch or tag to measure
!rm -rf /tmp/hivemind && git clone -q --depth 1 -b {REF} https://github.com/monochrome-muncher/hivemind /tmp/hivemind
!cp /tmp/hivemind/scripts/longmemeval-gpu/lme_gpu.py /tmp/lme_gpu.py
!cd /tmp/hivemind && uv sync -q && uv run python -c "import tests.eval.longmemeval; print('harness ok')"
!mkdir -p /tmp/lme && curl -sSL -o /tmp/lme/longmemeval_s_cleaned.json https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json && ls -la /tmp/lme
import transformers, torch
print("transformers", transformers.__version__, "torch", torch.__version__, "GPUs", torch.cuda.device_count())

In [ ]:
# The vector cache goes in /kaggle/working so each version's output keeps it;
# caches in any attached input (an earlier version's output) are merged in first.
import json, os
os.environ["LME_CACHE"] = "/kaggle/working/lme-cache"
os.environ["LME_SEED"] = "/kaggle/input"
TIME_BUDGET_HOURS = 10  # stop embedding after this; Kaggle ends a session at 12 hours
MAX_NEW_TEXTS = None  # on a rerun that reuses an earlier run's vectors, 1000: stop if they're missing
RUNS = [
 {
  "tag": "qwen3-8b",
  "model": "Qwen/Qwen3-Embedding-8B",
  "dim": 2000,
  "batch_tokens": 16384,
  "time_budget_hours": TIME_BUDGET_HOURS,
  "max_new_texts": MAX_NEW_TEXTS
 }
]
!free -g
# Exit status 137 means the system killed it, usually for running out of RAM.
!python /tmp/lme_gpu.py '{json.dumps(RUNS)}'; echo "lme_gpu.py exit status $?"
!ls -la /kaggle/working /kaggle/working/lme-cache/*